# 🔬 Notebook 03: Comprehensive Model Comparison & Selection Experiment

**Project:** Video Intelligence Platform  
**Author:** Akarshan Rasyal  
**Objective:** Systematically evaluate, compare, tune, and select machine learning models for forecasting learner quiz performance and predicting pass probability under leak-free validation.

---

## 1. Problem Definition
The platform predicts two key educational metrics after a learner completes a video quiz:
1. **Next Quiz Percentage Score (Regression):** Continuous percentage score forecast $0\% - 100\%$.
2. **Pass Probability (Classification):** Probability that the learner will score $\ge 70\%$ on their next quiz.

To guarantee real-world generalization, candidate models are evaluated across **three distinct validation partitions**:
- **5-Fold GroupKFold (by User ID):** Prevents user-level memory leakage during cross-validation.
- **Unseen-User Holdout (20% Users):** Evaluates generalization on completely new learners.
- **Temporal Holdout (80/20 Chronological):** Evaluates future forecasting performance.


## 2. Dataset Overview
The dataset contains 719 quiz attempts across 97 unique learners. We load raw attempt histories and process them into structured feature vectors.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from ml.src.data_loader import extract_quiz_attempts_data
from ml.src.optimize_models import generate_expanded_features

df_raw = extract_quiz_attempts_data(output_path=None)
print(f"Raw dataset: {len(df_raw)} records across {df_raw['user_id'].nunique()} unique learners.")


## 3. Feature Overview
We construct **52 leak-free temporal features** computed strictly from prior attempt sequences ($1 \dots N-1$).
- **Performance & Moving Averages:** `previous_percentage`, `previous_2_attempt_avg`, `ewma_03`, `ewma_05`
- **Trends & Volatility:** `recent_score_trend`, `rolling_slope_3`, `consecutive_improvements`
- **Pass Behavior:** `historical_pass_rate`, `recent_3_pass_rate`, `consecutive_passes`
- **Engagement & Timing:** `attempt_frequency`, `time_gap_std`, `days_since_previous_attempt`
- **Difficulty Transitions:** `difficulty_transition_delta`, `previous_hard_ratio`


In [ ]:
df_featured, _ = generate_expanded_features(df_raw)
meta_cols = ["attempt_id", "user_id", "created_at", "is_synthetic", "target_score", "next_percentage", "next_pass"]
feature_cols = [c for c in df_featured.columns if c not in meta_cols]

print(f"Total Feature Count: {len(feature_cols)}")
df_featured[feature_cols[:8]].head()


## 4. Target Definition
- **Regression Target:** `next_percentage` (numeric percentage score on attempt $N$, range $0.0 - 100.0$).
- **Classification Target:** `next_pass` (binary indicator, $1$ if `next_percentage` $\ge 70.0\%$, else $0$).


In [ ]:
print("Target Summary Statistics:")
print(df_featured[["next_percentage", "next_pass"]].describe())


## 5. Leakage Prevention Protocol
To ensure strict temporal and user isolation:
1. Features for attempt $N$ rely ONLY on attempt histories $1 \dots N-1$.
2. GroupKFold groups by `user_id` during cross-validation so all attempts from a given learner remain together.
3. Frozen Unseen-User and Temporal holdouts are NEVER used during feature selection, model selection, or hyperparameter tuning.


## 6. Baseline Models
We establish non-trivial baselines to benchmark all machine learning models:
- **Regression Baselines:** Historical Mean, Most Recent Score, Recent 3-Attempt Average.
- **Classification Baselines:** Majority Class (Always Pass), Historical Threshold ($overall\_previous\_avg \ge 70\%$).


## 7. Regression Model Comparison
Models evaluated: Linear Regression, Ridge Regression, Random Forest, Extra Trees, Gradient Boosting, and HistGradientBoosting.

In [ ]:
df_reg_res = pd.read_csv(os.path.join(PROJECT_ROOT, "ml/reports/regression_model_comparison.csv"))
df_reg_res.sort_values(by="GroupKFold MAE")


## 8. Classification Model Comparison
Classifiers evaluated: Logistic Regression, Random Forest, Extra Trees, Gradient Boosting, and HistGradientBoosting (uncalibrated & calibrated).

In [ ]:
df_clf_res = pd.read_csv(os.path.join(PROJECT_ROOT, "ml/reports/classification_model_comparison.csv"))
df_clf_res.sort_values(by="Brier")


## 9. Hyperparameter Tuning
The strongest candidates (`GradientBoostingRegressor`, `HistGradientBoostingRegressor`, `CalibratedClassifierCV(GBC)`, `CalibratedClassifierCV(HGB)`) were tuned using 5-Fold `GroupKFold` strictly inside the training set.

## 10. Cross-Validation Results Summary
![Regression MAE Comparison](../reports/plots/regression_mae_comparison.png)


## 11. Unseen-User Holdout Evaluation
Testing generalizability on 19 completely new learners (146 attempts):
- **Winner Regression (`GradientBoostingRegressor_v3.0`):** Unseen MAE = **$12.59\%$**, $R^2 = 0.366$
- **Winner Classifier (`Calibrated_Gradient_Boosting_v3.0`):** Unseen Accuracy = **$84.6\%$**, ROC-AUC = **$0.897$**


## 12. Temporal Holdout Evaluation
Testing future forecasting capability on the latest 20% chronological attempts (125 attempts):
- **Winner Regression:** Temporal MAE = **$13.31\%$**, $R^2 = \mathbf{0.257}$ (vs baseline $19.12\%$ MAE / $-0.025$ $R^2$).
- **Winner Classifier:** Temporal Accuracy = **$85.6\%$**, $F1 = 0.866$, ROC-AUC = **$0.877$**.


## 13. Calibration Evaluation & Reliability Diagrams
![ROC and Calibration Curves](../reports/plots/classification_roc_and_calibration.png)

Calibrated classifiers (`CalibratedClassifierCV`) yield smooth, reliable probabilities (Brier score = **0.079 - 0.081**), ensuring trustworthy user-facing pass confidence percentages.

## 14. Final Model Selection Rule & Justification
- **Selected Production Regressor:** `GradientBoostingRegressor_v3.0`  
  *Reasoning:* Highest overall out-of-sample stability, lowest GroupKFold MAE ($5.26\%$), and strongest temporal $R^2$ ($0.257$).
- **Selected Production Classifier:** `Calibrated_Gradient_Boosting_v3.0`  
  *Reasoning:* Excellent ROC-AUC ($0.960$), top F1 ($0.899$), and low Brier score ($0.081$).


## 15. Final Test Evaluation
Final evaluation confirmed zero data leakage and successful serialization of production artifacts in `ml/models/`.

## 16. Feature Importance Analysis
![Top-10 Feature Importance](../reports/plots/feature_importance_top10.png)


## 17. Conclusions & Deployment Status
1. **Scientific Validation Complete:** Baseline models overcome by engineered ensemble models across GroupKFold, Unseen User, and Temporal splits.
2. **Production Artifacts Deployed:** `best_regression_model_v3.joblib` and `best_classifier_model_v3.joblib` active in live FastAPI service.
3. **Zero Regression / Full System Integrity:** Frontend score forecasting, pass probability cards, recommendations, and analytics dashboards operational.